# AgroSmart AI — Análise exploratória de dados (EDA)

**Dataset:** Crop Recommendation Dataset (Kaggle, Atharva Ingle).

**Objetivo:** descrever o conjunto de dados, verificar sua qualidade, examinar a distribuição das classes, possíveis outliers e correlações. As figuras são geradas em `artigo/figuras/`.

> Este notebook **não treina modelos** e não faz pré-processamento ajustado aos dados completos. Imputação, escalonamento, seleção e ajuste de hiperparâmetros serão realizados **somente nos dados de treinamento** nas etapas posteriores, para evitar vazamento de dados.

Execute as células em ordem no VS Code. O CSV deve estar em `AgroSmart-AI/data/Crop_recommendation.csv`.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

# Localiza a raiz tanto se o notebook estiver aberto na pasta do projeto
# quanto se estiver sendo executado a partir de notebooks/.
base = Path.cwd().resolve()
if not (base / "data" / "Crop_recommendation.csv").exists():
    base = base.parent
csv_path = base / "data" / "Crop_recommendation.csv"
assert csv_path.exists(), f"CSV não encontrado: {csv_path}"
fig_dir = base / "artigo" / "figuras"
fig_dir.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(csv_path)
display(df.head())
print("Dimensões:", df.shape)

## 1. Estrutura, qualidade e estatísticas descritivas

In [ ]:
print("Colunas:", df.columns.tolist())
print("\nTipos:\n", df.dtypes)
print("\nValores ausentes por coluna:\n", df.isna().sum())
print("\nRegistros integralmente duplicados:", int(df.duplicated().sum()))
print("\nCulturas distintas:", df["label"].nunique())
display(df.describe().T.round(3))

## 2. Distribuição das classes (Figura 1)

Essa figura permite verificar se há classes minoritárias que afetariam a avaliação de classificação multiclasse.

In [ ]:
contagens = df["label"].value_counts().sort_index()
print(contagens.to_string())
fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(x=contagens.index, y=contagens.values, color="#42855B", ax=ax)
ax.set(title="Distribuição das culturas agrícolas", xlabel="Cultura", ylabel="Quantidade de registros")
ax.tick_params(axis="x", rotation=70)
fig.tight_layout()
fig.savefig(fig_dir / "figura_1_distribuicao_classes.png", dpi=180)
plt.show()

## 3. Correlações entre atributos (Figura 2)

Correlação de Pearson mostra associação linear entre atributos numéricos; **não demonstra causalidade**. Não incluímos a classe `label` nessa matriz.

In [ ]:
features = ["N", "P", "K", "temperature", "humidity", "ph", "rainfall"]
corr = df[features].corr(numeric_only=True)
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, center=0, square=True, ax=ax)
ax.set_title("Matriz de correlação dos atributos")
fig.tight_layout()
fig.savefig(fig_dir / "figura_2_correlacoes.png", dpi=180)
plt.show()

## 4. Inspeção de valores extremos (Figura 3)

Usamos boxplots padronizados **somente para visualização**. Os valores não serão removidos automaticamente: diferenças entre tipos de cultura podem produzir extremos legítimos.

In [ ]:
# Padronização apenas para visualização exploratória; NÃO será salva nem reutilizada no treinamento.
scaled_plot = (df[features] - df[features].mean()) / df[features].std(ddof=0)
fig, ax = plt.subplots(figsize=(11, 5))
sns.boxplot(data=scaled_plot, ax=ax)
ax.set(title="Distribuições padronizadas para inspeção visual", xlabel="Atributos", ylabel="Desvio-padrão em relação à média")
fig.tight_layout()
fig.savefig(fig_dir / "figura_3_boxplots.png", dpi=180)
plt.show()

## 5. Verificação inicial de extremos por IQR

Contagens por atributo (um mesmo registro pode aparecer em várias colunas). Não significa que esses casos sejam erros. É importante revisar por cultura antes de tomar decisões de limpeza.

In [ ]:
q1 = df[features].quantile(0.25)
q3 = df[features].quantile(0.75)
iqr = q3 - q1
flag = (df[features] < q1 - 1.5 * iqr) | (df[features] > q3 + 1.5 * iqr)
display(flag.sum().rename("Valores fora de 1,5×IQR").to_frame())
print("Registros com pelo menos um valor sinalizado:", int(flag.any(axis=1).sum()))

## 6. Observações para o artigo

Depois de executar, anote os resultados reais (não invente métricas):

- Quantidade de registros e culturas; distribuição e presença de ausentes/duplicatas.
- Atributos com associações mais fortes (descrever sem afirmar causalidade).
- Possíveis valores extremos e por que **não foram excluídos automaticamente**.
- Limitação: esta base representa relações contidas no dataset; o sistema não substitui avaliação agronômica local.

**Próxima etapa:** separar um conjunto de teste intocado, definir `Pipeline`/`ColumnTransformer` com ajuste apenas no treino e comparar `DummyClassifier`, `RandomForestClassifier` e `SVC` com validação cruzada estratificada.
